# C-02 · House-Keeping Diagnostics

Menura's HK (house-keeping) system records global energy budgets, magnetic field statistics, and particle counts every `rate_save_t_cst` iterations throughout the run.  These lightweight arrays are appended to files in `products/HK/` and give you a quick health-check without loading the large field snapshots.

All HK files are 1-D arrays (or 2-D for multi-component quantities like `B_mean`) with one entry per saved time step.  Files are written **per MPI rank**; to get global totals you sum across all ranks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# --- Configure ---
RUN_DIR  = '../menura'         # directory containing products/
PRODUCTS = os.path.join(RUN_DIR, 'products')
HK_DIR   = os.path.join(PRODUCTS, 'HK')

# Read parameters from products/parameters.txt
p = np.recfromtxt(os.path.join(PRODUCTS, 'parameters.txt'))
params = {t[0].decode('UTF-8'): float(t[1]) for t in p}

DT             = params['dt']
RATE_SAVE_T    = int(params['rate_save_t_cst'])
NB_PROC_Y      = int(params.get('mpi_nb_proc_y', params.get('mpi_nb_proc', 1)))
NB_PROC_Z      = int(params.get('mpi_nb_proc_z', 1))

DT_LOW = DT * RATE_SAVE_T    # time between HK saves

print(f'HK directory: {HK_DIR}')
print(f'HK saved every {RATE_SAVE_T} iterations  (dt_low = {DT_LOW:.3f} / Ω_ci)')
print(f'MPI grid: {NB_PROC_Y} × {NB_PROC_Z}')

## 1. Load simulation time

`simu_time.npy` is written by rank 0 only and contains the normalised simulation time at each HK save step.

In [ ]:
simu_time = np.load(os.path.join(HK_DIR, 'simu_time.npy'))
run_time  = np.load(os.path.join(HK_DIR, 'run_time.npy'))    # wall-clock seconds

print(f'Number of HK save points: {len(simu_time)}')
print(f'Time span: 0 → {simu_time[-1]:.2f} / Ω_ci')
print(f'Wall-clock time: {run_time[-1]/3600:.2f} h')

## 2. Load and sum energies across all MPI ranks

Each rank stores energy for its sub-domain.  The global energy is the sum over all ranks.

In [ ]:
def load_hk_sum(hk_name, hk_dir, nb_y, nb_z):
    """
    Load a HK scalar array and sum across all MPI ranks.
    
    Parameters
    ----------
    hk_name : str  — file prefix, e.g. 'energy_mag'
    hk_dir  : str  — path to products/HK/
    nb_y, nb_z : int
    
    Returns
    -------
    np.ndarray — summed time series
    """
    total = None
    for ry in range(nb_y):
        for rz in range(nb_z):
            fn = os.path.join(hk_dir, f'{hk_name}_rank_{ry}_{rz}.npy')
            arr = np.load(fn)
            total = arr if total is None else total + arr
    return total

e_mag  = load_hk_sum('energy_mag',  HK_DIR, NB_PROC_Y, NB_PROC_Z)
e_kin  = load_hk_sum('energy_kin',  HK_DIR, NB_PROC_Y, NB_PROC_Z)
e_elec = load_hk_sum('energy_elec', HK_DIR, NB_PROC_Y, NB_PROC_Z)

print(f'Energy arrays: {len(e_mag)} time steps')

## 3. Energy evolution plot

In [ ]:
t = simu_time[:len(e_mag)]   # align length in case run was interrupted

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(t, e_mag,  label='Magnetic', color='royalblue')
ax.plot(t, e_kin,  label='Kinetic',  color='tomato')
ax.plot(t, e_elec, label='Electric', color='gold')
ax.set_xlabel('Time [1/Ω_ci]')
ax.set_ylabel('Energy (normalised, summed over domain)')
ax.set_title('Energy evolution')
ax.legend()

ax = axes[1]
total = e_mag + e_kin + e_elec
ax.plot(t, total, color='black')
ax.set_xlabel('Time [1/Ω_ci]')
ax.set_ylabel('Total energy')
ax.set_title('Total energy (conservation check)')

plt.tight_layout()
plt.savefig('energy_evolution.png', dpi=150)
plt.show()

## 4. Active particle counts

`active_part_0` tracks solar wind ions; `active_part_1` tracks cometary/planetary secondary ions.

In [ ]:
def load_hk_stack(hk_name, hk_dir, nb_y, nb_z):
    """Load and stack HK arrays across ranks (returns 2D array [nb_ranks, nb_times])."""
    arrays = []
    for ry in range(nb_y):
        for rz in range(nb_z):
            fn = os.path.join(hk_dir, f'{hk_name}_rank_{ry}_{rz}.npy')
            arrays.append(np.load(fn))
    return np.stack(arrays, axis=0)   # shape: (nb_ranks, nb_times)

# Sum active particles across all ranks
sw_stack  = load_hk_stack('active_part_0', HK_DIR, NB_PROC_Y, NB_PROC_Z)
pla_stack = load_hk_stack('active_part_1', HK_DIR, NB_PROC_Y, NB_PROC_Z)

sw_total  = sw_stack.sum(axis=0)    # global SW particle count
pla_total = pla_stack.sum(axis=0)   # global secondary ion count

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t[:len(sw_total)],  sw_total,  label='Solar wind ions')
ax.plot(t[:len(pla_total)], pla_total, label='Planetary/cometary ions')
ax.set_xlabel('Time [1/Ω_ci]')
ax.set_ylabel('Active particle count')
ax.set_title('Active particle populations')
ax.legend()
plt.tight_layout()
plt.savefig('active_particles.png', dpi=150)
plt.show()

## 5. Magnetic field statistics

`B_mean` stores the domain-averaged B vector (shape `[nb_times, 3]`); `rms_B` stores the RMS of B fluctuations.

In [ ]:
def load_hk_sum_2d(hk_name, hk_dir, nb_y, nb_z):
    """Sum a 2D HK array (e.g. B_mean shape [nb_times, 3]) across ranks."""
    total = None
    for ry in range(nb_y):
        for rz in range(nb_z):
            fn = os.path.join(hk_dir, f'{hk_name}_rank_{ry}_{rz}.npy')
            arr = np.load(fn)
            total = arr if total is None else total + arr
    return total / (nb_y * nb_z)   # average (not sum) for mean quantities

B_mean = load_hk_sum_2d('B_mean', HK_DIR, NB_PROC_Y, NB_PROC_Z)  # shape (nb_times, 3)
rms_B  = load_hk_sum('rms_B', HK_DIR, NB_PROC_Y, NB_PROC_Z)

fig, axes = plt.subplots(1, 2, figsize=(14, 3))

ax = axes[0]
labels = ['Bx', 'By', 'Bz']
for i, lbl in enumerate(labels):
    ax.plot(t[:B_mean.shape[0]], B_mean[:, i], label=lbl)
ax.set_xlabel('Time [1/Ω_ci]')
ax.set_ylabel('Mean B (normalised)')
ax.set_title('Domain-averaged B components')
ax.legend()

ax = axes[1]
ax.plot(t[:len(rms_B)], rms_B, color='darkgreen')
ax.set_xlabel('Time [1/Ω_ci]')
ax.set_ylabel('RMS(δB)')
ax.set_title('RMS magnetic fluctuations')

plt.tight_layout()
plt.savefig('B_statistics.png', dpi=150)
plt.show()

## 6. Run-time performance

`run_time.npy` records wall-clock time (seconds) at each HK save point.

In [ ]:
# Iteration throughput: iterations per second
n_it_per_save = RATE_SAVE_T
dt_wall = np.diff(run_time)   # seconds per HK interval
throughput = n_it_per_save / dt_wall   # iterations / second

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t[1:len(throughput)+1], throughput)
ax.set_xlabel('Time [1/Ω_ci]')
ax.set_ylabel('Iterations / second')
ax.set_title('Solver throughput')
plt.tight_layout()
plt.show()

print(f'Mean throughput: {throughput.mean():.2f} it/s')
print(f'Total wall time: {run_time[-1]/60:.1f} min')

## 7. Exercises

1. What physical process would cause the magnetic energy to grow steadily while the kinetic energy decreases?  What if both grow?
2. Menura does not conserve total energy exactly (it uses a resistive, open-boundary solver).  How would you quantify the energy budget error over the run?
3. If `active_part_1` grows monotonically, what does that indicate about the secondary ion injection rate?
4. Plot `rms_B` against `e_mag`.  What relationship do you expect?
5. Modify `load_hk_sum` to work for 1-rank runs where there is no `_rank_Y_Z` suffix — or does Menura always write the rank suffix even for single-process runs?  Check `functions_o.cpp` to verify.